In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Paper 4 Test: Ephaptic Coupling + Mechanotransduction

Paper 4 specifies THREE coupling mechanisms beyond synapses:

1. Ephaptic coupling (p.27, 35):
   E_eph ~ V/d ~ mV/mm (endogenous electric fields)
   DeltaV_m = -integral E_eph . dl ~ E_eph * L_cell
   E_eph ~ 1-5 mV/mm, L_cell ~ 100 um â†’ DeltaV ~ 0.1-0.5 mV
   This shifts spike timing by 1-5 ms.
   Resonant amplification: G(omega) = G_0/sqrt((omega_0^2-omega^2)^2 + (gamma*omega)^2)
   Q factor: Q = omega_0/gamma ~ 2-10

2. Mechanotransduction (p.39):
   H_adhesion = sum (1/2)m_eff r_dot^2 + (1/2)k_adh(r-r0)^2 + V_coupling
   m_eff ~ 10^-15 kg, k_adh ~ 0.1-1 pN/nm
   Stress waves: d^2u/dt^2 = c_mech^2 nabla^2 u - gamma_damp du/dt
   c_mech = sqrt(E_tissue/rho) ~ 10-100 m/s

3. Biophotonic waveguides (p.27):
   Myelin sheaths as optical cladding:
   beta = (omega/c) * sqrt(eps_core - eps_cladding)

In [ ]:
import json

In [ ]:
import numpy as np

In [ ]:
FINDINGS = []

In [ ]:
def add_finding(tag, description, data):
    FINDINGS.append({"tag": tag, "description": description, "data": data})
    print(f"[FINDING] {tag}: {description}")
    for k, v in data.items():
        print(f"  {k}: {v}")

In [ ]:
def order_param(theta):
    z = np.mean(np.exp(1j * theta))
    return np.abs(z), np.angle(z)

In [ ]:
# --- Test 1: Ephaptic coupling strength (Paper 4 values) ---
print("=== Test 1: Ephaptic coupling â€” membrane potential shift ===")

In [ ]:
# Paper 4 parameters (p.35)
E_eph_values = [1, 2, 3, 5]  # mV/mm
L_cell = 100e-3  # mm (100 um)

In [ ]:
for E in E_eph_values:
    DeltaV = E * L_cell  # mV
    # Spike timing shift: ~1-5 ms for 0.1-0.5 mV
    # Approximate: dt_spike ~ DeltaV / (dV/dt at threshold)
    # dV/dt at threshold ~ 10 mV/ms (typical action potential)
    dVdt_threshold = 10  # mV/ms
    dt_spike = DeltaV / dVdt_threshold  # ms
    print(f"  E_eph={E} mV/mm â†’ DeltaV={DeltaV:.2f} mV â†’ dt_spike={dt_spike:.2f} ms")

In [ ]:
# Resonant amplification
omega_0 = 2 * np.pi * 40  # 40 Hz gamma
gamma_damp = omega_0 / 5  # Q=5
freqs = np.linspace(1, 100, 200)
omega = 2 * np.pi * freqs
G = 1.0 / np.sqrt((omega_0**2 - omega**2) ** 2 + (gamma_damp * omega) ** 2)
G_norm = G / np.max(G)

In [ ]:
# Peak gain and bandwidth
peak_freq = freqs[np.argmax(G_norm)]
half_power = np.where(G_norm > 0.707)[0]
bandwidth = freqs[half_power[-1]] - freqs[half_power[0]] if len(half_power) > 1 else 0

In [ ]:
add_finding(
    "EPHAPTIC_COUPLING",
    "Ephaptic field coupling (Paper 4 values)",
    {
        "E_eph_range_mV_mm": "1-5",
        "DeltaV_range_mV": "0.1-0.5",
        "spike_timing_shift_ms": "0.01-0.05",
        "resonant_peak_Hz": round(peak_freq, 1),
        "Q_factor": round(omega_0 / gamma_damp, 1),
        "bandwidth_Hz": round(bandwidth, 1),
        "equation": "Paper 4, p.35: DeltaV_m = E_eph * L_cell, G(omega) = G_0/sqrt(...)",
    },
)

In [ ]:
# --- Test 2: Ephaptic coupling in Kuramoto model ---
print("\n=== Test 2: Ephaptic K_ij adds long-range sync ===")

In [ ]:
N = 100
np.random.seed(42)
omega_neural = 2 * np.pi * 40  # gamma band
sigma_w = omega_neural * 0.05
omegas = np.random.normal(omega_neural, sigma_w, N)

In [ ]:
# Spatial arrangement (2D cortical sheet)
positions = np.random.rand(N, 2) * 5  # 5mm x 5mm patch
distances = np.sqrt(np.sum((positions[:, None] - positions[None, :]) ** 2, axis=2))

In [ ]:
# Synaptic coupling (short range, ~300 um)
K_syn = 2.0
lambda_syn = 0.3  # mm
K_syn_matrix = K_syn * np.exp(-distances / lambda_syn)
np.fill_diagonal(K_syn_matrix, 0)

In [ ]:
# Ephaptic coupling (longer range, ~1 mm, 1/r^2 field)
K_eph = 0.3  # weaker but longer range
K_eph_matrix = K_eph / (1 + (distances / 1.0) ** 2)
np.fill_diagonal(K_eph_matrix, 0)

In [ ]:
theta_syn = np.random.uniform(0, 2 * np.pi, N)
theta_both = theta_syn.copy()

In [ ]:
dt = 1e-4
r_syn_trace = []
r_both_trace = []

In [ ]:
for step in range(30000):
    # Synaptic only
    dtheta_s = omegas.copy()
    for i in range(N):
        dtheta_s[i] += np.sum(K_syn_matrix[i] * np.sin(theta_syn - theta_syn[i])) / N
    theta_syn += dt * dtheta_s

    # Synaptic + ephaptic
    dtheta_b = omegas.copy()
    for i in range(N):
        dtheta_b[i] += (
            np.sum((K_syn_matrix[i] + K_eph_matrix[i]) * np.sin(theta_both - theta_both[i])) / N
        )
    theta_both += dt * dtheta_b

    if step % 300 == 0:
        r_s, _ = order_param(theta_syn)
        r_b, _ = order_param(theta_both)
        r_syn_trace.append(float(r_s))
        r_both_trace.append(float(r_b))

In [ ]:
add_finding(
    "EPHAPTIC_SYNC",
    "Ephaptic coupling extends synchronisation range",
    {
        "r_synaptic_only": round(float(np.mean(r_syn_trace[-20:])), 4),
        "r_synaptic_plus_ephaptic": round(float(np.mean(r_both_trace[-20:])), 4),
        "enhancement_percent": round(
            float((np.mean(r_both_trace[-20:]) / max(np.mean(r_syn_trace[-20:]), 0.01) - 1) * 100),
            1,
        ),
        "K_syn": K_syn,
        "K_eph": K_eph,
        "lambda_syn_mm": lambda_syn,
        "note": "ephaptic fields enable long-range sync beyond synaptic reach",
    },
)

In [ ]:
# --- Test 3: Mechanical stress-phase coupling ---
print("\n=== Test 3: Mechanotransduction stress waves (Paper 4, p.39) ===")

Paper 4: c_mech = sqrt(E_tissue/rho) ~ 10-100 m/s
Compare: synaptic conduction ~ 1-100 m/s, ephaptic ~ speed of light in tissue

In [ ]:
E_tissue = 1e3  # Pa (brain tissue Young's modulus ~ 1 kPa)
rho_tissue = 1050  # kg/m^3
c_mech = np.sqrt(E_tissue / rho_tissue)

In [ ]:
# Paper 4: k_adh ~ 0.1-1 pN/nm, m_eff ~ 10^-15 kg
k_adh = 0.5e-3  # N/m (0.5 pN/nm)
m_eff = 1e-15  # kg
omega_mech = np.sqrt(k_adh / m_eff)
f_mech = omega_mech / (2 * np.pi)

In [ ]:
# 1D stress wave propagation
Nx = 100
dx = 50e-6  # 50 um spacing
gamma_mech = 10  # damping

In [ ]:
# CFL condition
dt_mech = 0.5 * dx / c_mech
T_mech = 0.01  # 10 ms
steps_mech = int(T_mech / dt_mech)

In [ ]:
u = np.zeros(Nx)  # displacement
v = np.zeros(Nx)  # velocity
# Impulse at left end
u[0] = 1e-6  # 1 um displacement

In [ ]:
u_trace = []
for step in range(min(steps_mech, 10000)):
    # Finite difference wave equation
    d2u = np.zeros(Nx)
    d2u[1:-1] = (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2
    d2u[0] = (u[1] - u[0]) / dx**2
    d2u[-1] = (u[-2] - u[-1]) / dx**2

    a = c_mech**2 * d2u - gamma_mech * v
    v += dt_mech * a
    u += dt_mech * v

    if step % 100 == 0:
        u_trace.append(u.copy())

In [ ]:
# Wave arrival time at far end
if len(u_trace) > 1:
    expected_time = (Nx * dx) / c_mech
else:
    expected_time = float("nan")

In [ ]:
add_finding(
    "MECHANOTRANSDUCTION",
    "Stress wave propagation in tissue",
    {
        "c_mech_m_s": round(float(c_mech), 3),
        "E_tissue_Pa": E_tissue,
        "rho_kg_m3": rho_tissue,
        "adhesion_f_resonance_Hz": round(float(f_mech), 0),
        "wave_travel_time_5mm_ms": round(float(5e-3 / c_mech * 1000), 3),
        "comparison": {
            "synaptic_conduction_m_s": "1-100",
            "ephaptic_field_m_s": "~10^8",
            "mechanical_wave_m_s": round(float(c_mech), 1),
        },
        "equation": "Paper 4, p.39: c_mech = sqrt(E/rho), d2u/dt2 = c^2 nabla^2 u - gamma du/dt",
    },
)

In [ ]:
# --- Test 4: Biophotonic waveguides (myelin as cladding) ---
print("\n=== Test 4: Myelin as biophotonic waveguide (Paper 4, p.27) ===")

Paper 4: beta = (omega/c) * sqrt(eps_core - eps_cladding)
Axon core: n ~ 1.38 (cytoplasm refractive index)
Myelin cladding: n ~ 1.44 (lipid bilayer, higher RI!)
Wait â€” for waveguide: core must have HIGHER n than cladding
Actually myelin has higher n â†’ this is like a hollow-core fibre
Alternative: consider unmyelinated segment as core, myelin as cladding
Or: cytoplasm n=1.38 vs extracellular n=1.335

In [ ]:
n_core = 1.38  # axon cytoplasm
n_cladding = 1.335  # extracellular fluid
# This gives total internal reflection for shallow angles

In [ ]:
# Biophoton wavelength: 200-800 nm (Paper 4, p.21)
wavelengths = np.array([200, 400, 500, 600, 800]) * 1e-9  # m
c_light = 3e8

In [ ]:
# Propagation constant
for lam in wavelengths:
    omega_ph = 2 * np.pi * c_light / lam
    if n_core > n_cladding:
        beta = (omega_ph / c_light) * np.sqrt(n_core**2 - n_cladding**2)
        # Number of modes: V = (2*pi*a/lambda) * NA, where NA = sqrt(n1^2 - n2^2)
        a_axon = 0.5e-6  # 0.5 um radius
        NA = np.sqrt(n_core**2 - n_cladding**2)
        V = (2 * np.pi * a_axon / lam) * NA
        n_modes = max(1, int(V**2 / 2))  # approximate
        print(f"  lambda={lam * 1e9:.0f}nm: beta={beta:.2e}/m, V={V:.2f}, modes~{n_modes}")

In [ ]:
# Attenuation in tissue
alpha_tissue = 10  # 1/cm (approximate for 500nm in tissue)
L_decay = 1.0 / alpha_tissue  # cm
L_decay_um = L_decay * 1e4  # um

In [ ]:
add_finding(
    "BIOPHOTONIC_WAVEGUIDE",
    "Myelin/axon as optical waveguide",
    {
        "n_core_cytoplasm": n_core,
        "n_cladding_extracellular": n_cladding,
        "NA": round(float(np.sqrt(n_core**2 - n_cladding**2)), 4),
        "V_parameter_500nm": round(
            float((2 * np.pi * 0.5e-6 / 500e-9) * np.sqrt(n_core**2 - n_cladding**2)), 3
        ),
        "decay_length_um": round(L_decay_um, 0),
        "biophoton_rate_per_cell": "10^-3/s (Paper 4, p.17)",
        "plausibility": "waveguide physics valid; signal strength is the bottleneck",
        "equation": "Paper 4, p.27: beta = (omega/c) sqrt(eps_core - eps_cladding)",
    },
)

In [ ]:
# --- Test 5: Developmental K evolution (Paper 4, p.47) ---
print("\n=== Test 5: Developmental coupling maturation (Paper 4, p.47) ===")

Paper 4: K_dev(t) = K_max * (1 - exp(-t/tau_mat)) * sigmoid((t_crit - t)/sigma_crit)
tau_maturation ~ 2-4 weeks
t_critical = species-dependent critical period

In [ ]:
K_max = 5.0
tau_mat = 3 * 7 * 24  # 3 weeks in hours
t_critical = 6 * 30 * 24  # ~6 months in hours
sigma_crit = 2 * 30 * 24  # 2 months width

In [ ]:
ages_months = np.linspace(0, 24, 100)  # 0-24 months
ages_hours = ages_months * 30 * 24

In [ ]:
K_dev = (
    K_max
    * (1 - np.exp(-ages_hours / tau_mat))
    / (1 + np.exp((ages_hours - t_critical) / sigma_crit))
)

In [ ]:
# Simulate network sync at different developmental stages
dev_sync = []
N_dev = 50
omega_dev = 2 * np.pi * 10
omegas_dev = np.random.normal(omega_dev, omega_dev * 0.1, N_dev)

In [ ]:
for age_idx in range(0, len(ages_months), 10):
    K_age = K_dev[age_idx]
    theta_d = np.random.uniform(0, 2 * np.pi, N_dev)
    dt = 0.001
    for step in range(5000):
        z = np.mean(np.exp(1j * theta_d))
        r = np.abs(z)
        psi = np.angle(z)
        dtheta = omegas_dev + K_age * r * np.sin(psi - theta_d)
        theta_d += dt * dtheta
    r_d, _ = order_param(theta_d)
    dev_sync.append(
        {
            "age_months": round(float(ages_months[age_idx]), 1),
            "K": round(float(K_age), 3),
            "r": round(float(r_d), 4),
        }
    )

In [ ]:
add_finding(
    "DEVELOPMENTAL_K",
    "Coupling maturation during development",
    {
        "results": dev_sync,
        "K_max": K_max,
        "tau_maturation_weeks": 3,
        "t_critical_months": 6,
        "equation": "Paper 4, p.47: K = K_max * (1-exp(-t/tau)) * sigmoid((t_c-t)/sigma)",
        "clinical": "disrupted critical period â†’ lasting sync deficits (autism, schizophrenia)",
    },
)

In [ ]:
# --- Output ---
print("\n" + "=" * 60)
print(f"TOTAL FINDINGS: {len(FINDINGS)}")
print("=" * 60)
output = {"notebook": "p4_ephaptic_mechanotransduction", "findings": FINDINGS}
print("\n--- JSON OUTPUT ---")
print(json.dumps(output, indent=2, default=str))